<a href="https://colab.research.google.com/github/RVbringiton/Metode-Numeris-/blob/main/Metode_Numeris.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

def fixpt(g, x0, Ea=1.e-7, maxit=30):
    """
    This function solves x=g(x) using fixed-point iteration.
    The method is repeated until either the relative error
    falls below Ea (default 1.e-7) or reaches maxit (default 30).
    Input:
        g     = name of the function for g(x)
        x0    = initial guess for x
        Ea    = relative error threshold
        maxit = maximum number of iterations
    Output:
        x1    = solution estimate
        ea    = relative error
        i+1   = number of iterations
    """
    for i in range(maxit):
        x1 = g(x0)
        ea = abs((x1 - x0) / x1)
        if ea < Ea:
            break
        x0 = x1
    return x1, ea, i+1

# ---------------------------------------------------------------------
# Implementasi Kasus Biomedis: Terapi Kanker Hipertermia
# Persamaan Asli: f(x) = (100/x) * e^(-0.5x) - 37 = 0
# Bentuk g(x):    g(x) = (100/37) * e^(-0.5x)
# ---------------------------------------------------------------------

def g(x):
    return (100 / 37) * np.exp(-0.5 * x)

# Parametrisasi Iterasi
x0 = 1.0          # Initial guess di dalam interval konvergen [1, 2]
Ea_target = 1.e-5 # Ambang batas relatif error

# Eksekusi Metode Fixed-Point Iteration
(xsoln, ea, n) = fixpt(g, x0, Ea=Ea_target)

# Cetak Hasil Pengerjaan
print("=" * 45)
print("  HASIL METODE SIMPLE FIXED-POINT ITERATION")
print("=" * 45)
print('Solution             = {0:8.5g} cm'.format(xsoln))
print('Relative error       = {0:8.3e}'.format(ea))
print('Number of iterations = {0:5d}'.format(n))
print("=" * 45)

  HASIL METODE SIMPLE FIXED-POINT ITERATION
Solution             =   1.3655 cm
Relative error       = 7.393e-06
Number of iterations =    30


In [ ]:
import numpy as np

def f(x):
    return (100 / x) * np.exp(-0.5 * x) - 37

def df(x):
    return -100 * np.exp(-0.5 * x) / (x**2) - 50 * np.exp(-0.5 * x) / x

x = 1.0
for i in range(1, 6):
    x = x - f(x) / df(x)
    print(f"Iterasi {i}: x = {x:.5f}")

Iterasi 1: x = 1.25998
Iterasi 2: x = 1.35636
Iterasi 3: x = 1.36542
Iterasi 4: x = 1.36549
Iterasi 5: x = 1.36549


In [ ]:
import numpy as np

def newtraph(f, df, x0, Ea=1.e-7, maxit=30):
    """
    Solves f(x) = 0 using Newton-Raphson method.
    Input:
        f     = function f(x)
        df    = derivative of f(x), i.e., f'(x)
        x0    = initial guess
        Ea    = relative error threshold
        maxit = maximum number of iterations
    Output:
        x1    = solution estimate
        ea    = relative error
        i+1   = number of iterations
    """
    for i in range(maxit):
        x1 = x0 - f(x0) / df(x0)
        ea = abs((x1 - x0) / x1)
        if ea < Ea:
            break
        x0 = x1
    return x1, ea, i+1

# ---------------------------------------------------------------------
# Implementasi Kasus Biomedis: Terapi Kanker Hipertermia
# ---------------------------------------------------------------------

def f(x):
    return (100 / x) * np.exp(-0.5 * x) - 37

def df(x):
    return -(100 * np.exp(-0.5 * x) / x) * ((1 / x) + 0.5)

# Parametrisasi Iterasi
x0 = 1.0          # Initial guess
Ea_target = 1.e-5 # Ambang batas relative error

# Eksekusi Metode Newton-Raphson
(xsoln, ea, n) = newtraph(f, df, x0, Ea=Ea_target)

# Cetak Hasil Pengerjaan
print("=" * 45)
print("  HASIL METODE NEWTON-RAPHSON")
print("=" * 45)
print('Solution             = {0:8.5g} cm'.format(xsoln))
print('Relative error       = {0:8.3e}'.format(ea))
print('Number of iterations = {0:5d}'.format(n))
print("=" * 45)

  HASIL METODE NEWTON-RAPHSON
Solution             =   1.3655 cm
Relative error       = 2.937e-09
Number of iterations =     5


In [1]:
"""
Tugas Metode Numerik: Metode Iteratif Jacobi & Gauss-Seidel
Kasus : Difusi oksigen tunak (steady state) pada 3 lapisan jaringan
        dengan konsumsi oksigen orde-1 (reaksi-difusi 1D, beda hingga)
"""
import numpy as np

# ==========================================================
# 1. DEFINISI MASALAH
# ==========================================================
# Persamaan: D * d2C/dx2 = k(x) * C     (C = tekanan parsial O2, mmHg)
# Diskretisasi beda hingga pusat (4 interval, 3 titik dalam):
#   -C[i-1] + (2 + lam_i) * C[i] - C[i+1] = 0 ,  lam_i = k_i * h^2 / D
D  = 2.0e-5                          # difusivitas O2 di jaringan (cm^2/s)
L  = 0.30                            # tebal total jaringan (cm)
h  = L / 4                           # jarak antar titik (cm)
k  = np.array([0.005, 0.007, 0.010]) # laju konsumsi O2 tiap lapisan (1/s)
C0 = 100.0                           # pO2 di sisi kapiler  (mmHg)
C4 = 40.0                            # pO2 di sisi seberang (mmHg)

lam = k * h**2 / D
A = np.array([[2 + lam[0], -1.0,       0.0      ],
              [-1.0,       2 + lam[1], -1.0     ],
              [ 0.0,       -1.0,       2 + lam[2]]])
b = np.array([C0, 0.0, C4])          # suku batas pindah ke ruas kanan

# ==========================================================
# 2. PARAMETER ITERASI
# ==========================================================
x0       = np.array([50.0, 50.0, 50.0])  # tebakan awal (mmHg)
TOL      = 1e-4                          # toleransi: max |x_baru - x_lama|
MAX_ITER = 50                            # iterasi maksimum

# ==========================================================
# 3. IMPLEMENTASI METODE
# ==========================================================
def cek_dominan(A):
    d = np.abs(np.diag(A))
    s = np.sum(np.abs(A), axis=1) - d
    return bool(np.all(d > s))

def jacobi(A, b, x0, tol, maxit):
    n = len(b); x = x0.copy(); riwayat = []
    for k_it in range(1, maxit + 1):
        xb = np.zeros(n)
        for i in range(n):
            s = sum(A[i, j] * x[j] for j in range(n) if j != i)
            xb[i] = (b[i] - s) / A[i, i]      # semua pakai nilai LAMA
        err = np.max(np.abs(xb - x))
        riwayat.append((k_it, *xb, err))
        x = xb
        if err < tol:
            return x, k_it, riwayat
    return x, maxit, riwayat

def gauss_seidel(A, b, x0, tol, maxit):
    n = len(b); x = x0.copy(); riwayat = []
    for k_it in range(1, maxit + 1):
        x_lama = x.copy()
        for i in range(n):
            s = sum(A[i, j] * x[j] for j in range(n) if j != i)
            x[i] = (b[i] - s) / A[i, i]       # langsung pakai nilai TERBARU
        err = np.max(np.abs(x - x_lama))
        riwayat.append((k_it, *x, err))
        if err < tol:
            return x, k_it, riwayat
    return x, maxit, riwayat

def cetak(nama, riwayat):
    print(f"\n=== Iterasi {nama} ===")
    print(f"{'k':>3} {'C1':>10} {'C2':>10} {'C3':>10} {'error':>12}")
    for r in riwayat:
        print(f"{r[0]:>3} {r[1]:>10.6f} {r[2]:>10.6f} {r[3]:>10.6f} {r[4]:>12.2e}")

# ==========================================================
# 4. EKSEKUSI & OUTPUT
# ==========================================================
if __name__ == "__main__":
    print("lambda_i =", np.round(lam, 5))
    print("Matriks A =\n", np.round(A, 5))
    print("Vektor b  =", b)
    print("Dominan diagonal :", cek_dominan(A))
    print("Tebakan awal:", x0, "| TOL:", TOL, "| Maks iterasi:", MAX_ITER)

    xj, kj, rj = jacobi(A, b, x0, TOL, MAX_ITER)
    cetak("Jacobi", rj)
    print(f"\nSolusi Jacobi       : C1={xj[0]:.4f}, C2={xj[1]:.4f}, C3={xj[2]:.4f} mmHg ({kj} iterasi)")

    xg, kg, rg = gauss_seidel(A, b, x0, TOL, MAX_ITER)
    cetak("Gauss-Seidel", rg)
    print(f"\nSolusi Gauss-Seidel : C1={xg[0]:.4f}, C2={xg[1]:.4f}, C3={xg[2]:.4f} mmHg ({kg} iterasi)")

    print("\nSolusi eksak (numpy.linalg.solve):", np.round(np.linalg.solve(A, b), 4))


lambda_i = [1.40625 1.96875 2.8125 ]
Matriks A =
 [[ 3.40625 -1.       0.     ]
 [-1.       3.96875 -1.     ]
 [ 0.      -1.       4.8125 ]]
Vektor b  = [100.   0.  40.]
Dominan diagonal : True
Tebakan awal: [50. 50. 50.] | TOL: 0.0001 | Maks iterasi: 50

=== Iterasi Jacobi ===
  k         C1         C2         C3        error
  1  44.036697  25.196850  18.701299     3.13e+01
  2  36.755039  15.807999  13.547397     9.39e+00
  3  33.998679  12.674630  11.596467     3.13e+00
  4  33.078790  11.488541  10.945378     1.19e+00
  5  32.730581  11.092704  10.698918     3.96e-01
  6  32.614372  10.942866  10.616666     1.50e-01
  7  32.570383  10.892860  10.585531     5.00e-02
  8  32.555702  10.873931  10.575140     1.89e-02
  9  32.550145  10.867614  10.571206     6.32e-03
 10  32.548290  10.865222  10.569894     2.39e-03
 11  32.547588  10.864424  10.569397     7.98e-04
 12  32.547354  10.864122  10.569231     3.02e-04
 13  32.547265  10.864021  10.569168     1.01e-04
 14  32.547236  10.86